In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import RFE
import matplotlib.pyplot as plt

df = pd.read_excel('data/raw.xlsx')

In [2]:
df.fillna(df.mean(), inplace=True)

In [3]:
df = pd.read_csv('data/processed.csv')

In [4]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_preprocessing_pipeline

preprocessing_pipeline = create_preprocessing_pipeline()
preprocessing_pipeline

Pipeline(steps=[('scaler', DataFrameScaler()),
                ('features_engineering_volumetric',
                 FeaturesEngineeringVolumetricSurfaceMolecule()),
                ('features_engineering_density',
                 FeaturesEngineeringDensityMorganFingerprints()),
                ('features_engineering_chi', FeaturesEngineeringChiIndices()),
                ('features_kappa', FeaturesEngineeringKappa()),
                ('features_bcut', FeaturesEngineeringBCUT())])

In [5]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['IC50, mM']

# Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = preprocessing_pipeline.fit_transform(X_train, y_train)
X_test = preprocessing_pipeline.transform(X_test)

In [6]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")

Mean Absolute Error: 295.78139631018536
Mean Squared Error: 263745.84765637165
R^2 Score: 0.2092982899659852


In [7]:
# Использование SelectKBest для отбора лучших признаков
selector = SelectKBest(score_func=f_regression, k=55)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Получение индексов выбранных признаков
selected_features = X_train.columns[selector.get_support()]

print("Выбранные признаки:", selected_features)

# Обучение модели линейной регрессии на выбранных признаках
model = LinearRegression()
model.fit(X_train_selected, y_train)

# Предсказание
y_pred = model.predict(X_test_selected)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")

[False False False  True False False  True  True False False  True  True
  True  True False  True False  True  True False False False False  True
 False  True  True  True  True  True  True  True  True  True  True  True
  True False False  True  True False  True False False False False False
 False False False False False  True  True False False False False False
 False  True  True False False False False False False False False False
  True  True False False False False False False False False False  True
  True False False  True False False False False False  True False False
  True  True False False False False  True False  True False False False
 False False  True  True  True False False  True  True False False False
 False False False  True False False False False False False False False
  True False False False False  True False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False Fa

In [8]:
# Создание модели линейной регрессии
model = LinearRegression()

# Использование RFE для отбора признаков
rfe = RFE(estimator=model, n_features_to_select=55)
X_train_selected = rfe.fit_transform(X_train, y_train)
X_test_selected = rfe.transform(X_test)

# Получение индексов выбранных признаков
selected_features = X_train.columns[rfe.support_]

print("Выбранные признаки:", selected_features)

# Обучение модели линейной регрессии на выбранных признаках
model.fit(X_train_selected, y_train)

# Предсказание
y_pred = model.predict(X_test_selected)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2}")

Выбранные признаки: Index(['MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge',
       'MinAbsPartialCharge', 'FpDensityMorgan1', 'FpDensityMorgan2',
       'BCUT2D_CHGHI', 'BCUT2D_LOGPHI', 'Chi0', 'Chi1n', 'Chi1v', 'Chi2n',
       'Chi2v', 'Chi3n', 'Chi3v', 'Chi4v', 'Kappa2', 'LabuteASA', 'PEOE_VSA1',
       'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'PEOE_VSA14',
       'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6',
       'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'SMR_VSA1', 'SMR_VSA3',
       'SMR_VSA6', 'SlogP_VSA12', 'SlogP_VSA2', 'TPSA', 'EState_VSA1',
       'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5',
       'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'NHOHCount',
       'NOCount', 'NumHAcceptors', 'NumHDonors', 'NumHeteroatoms',
       'fr_halogen', 'Chi_sum', 'Kappa'],
      dtype='object')
Mean Absolute Error: 273.69575656140273
Mean Squared Error: 221476.789552807
R^2 Score: 0.33601958935705933
